# 00 · Datos y features

Descarga el panel de mercado, construye los 20 canales causales de `X` y las series objetivo, y deja constancia de con qué datos exactos se trabajó.

**Responsable:** Oscar

**Entradas**

- `data/catalog.yaml`

**Salidas**

- `data/raw/precios.parquet`
- `data/processed/canales.parquet`
- `data/processed/objetivos.parquet`

**Tiempo estimado:** ~5 min la primera vez (la descarga de yfinance domina); <15 s con la caché ya escrita.

In [ ]:
import sys; sys.path.insert(0, "..")   # permite ejecutar desde notebooks/
import src                              # fija el backend de Keras a PyTorch
from src import config, viz
config.fijar_semillas()
viz.aplicar_estilo()

In [ ]:
import pandas as pd

from src import datos, features

## Qué gobierna este notebook

`data/catalog.yaml` es la fuente de verdad. Ningún ticker, fecha ni longitud de
ventana está escrito aquí: todo se lee del catálogo. Cambiar un parámetro allí y
volver a ejecutar este notebook y el 02 regenera el dataset completo.

In [ ]:
catalogo = config.cargar_catalogo()
periodo = catalogo["periodo"]

print("Periodo:", periodo["inicio"], "→", periodo["fin"])
print("Activos:", len(config.tickers()), "· canales:", config.n_canales())
pd.DataFrame(catalogo["universo"])

## Descarga

Los precios se bajan una sola vez y quedan cacheados en `data/raw/precios.parquet`.
yfinance no es una fuente inmutable: revisa su histórico y ajusta dividendos hacia
atrás, de modo que dos descargas separadas en el tiempo no dan lo mismo. Fijar la
caché es lo que hace que los resultados del taller sean reproducibles entre los
tres integrantes.

`forzar=True` solo se usa para ampliar la ventana temporal, y entonces hay que
reejecutar todo el pipeline.

In [ ]:
precios = datos.descargar_precios()
print("Panel:", precios.shape)
precios.tail(3)

## Control de calidad

Esta tabla es el registro de qué datos se usaron. Si un compañero obtiene otras
métricas, lo primero que hay que comparar es esta salida.

In [ ]:
datos.resumen(precios)

## Política de huecos: recortar, nunca imputar

El panel se recorta al tramo en que **todas** las series tienen dato. Rellenar un
hueco con el valor anterior inventaría un movimiento de precio nulo que no
ocurrió, y eso contamina directamente la volatilidad realizada, que es una de las
dos variables objetivo del taller.

En la práctica el recorte lo gobiernan los ETF de renta fija (TLT/IEF/LQD, desde
2002-07) y elimina los días de festivo parcial en que una bolsa abre y otra no.

In [ ]:
print("Valores ausentes tras el alineado:", int(precios.isna().sum().sum()))
print("Primer día común:", precios.index[0].date())
print("Último día común: ", precios.index[-1].date())

salto = precios.index.to_series().diff().dt.days
print("Saltos de más de 5 días naturales:", int((salto > 5).sum()))

## Construcción de los canales

Todas las transformaciones son causales: el valor en `t` solo usa información de
`t` o anterior. El error clásico en series financieras es el z-score sobre la
muestra completa, que mejora las métricas porque el modelo conoce la media y la
desviación de todo el histórico, futuro incluido.

El orden de las columnas es un contrato con `data/processed/`: reordenarlo
invalida los generadores ya entrenados.

In [ ]:
canales = features.construir_canales(precios)

print("Canales:", canales.shape)
print("Rango:", canales.index[0].date(), "→", canales.index[-1].date())
print("Sesiones perdidas al arrancar:", len(precios) - len(canales))
canales.describe().T

## Contraste de causalidad

`features.assert_causal` aplica la transformación a la serie completa y a un
prefijo de esa misma serie, y exige que los valores del prefijo no cambien.
Es el contraste que delata el error clásico —el z-score sobre la muestra
entera—: si la transformación usa estadísticos del futuro, conocerlo reescribe
retroactivamente los valores del pasado y la comparación falla.

Se aplica sobre las primitivas y su serie **sin recortar**, no sobre `canales`,
que ya tiene el arranque eliminado. Con esta formulación entran también
`drawdown_sp500` y `dispersion_sectorial`, que el contraste anterior dejaba
fuera: son causales, pero están definidas desde la primera observación (el
máximo expanding arranca con `min_periods=1` y la dispersión transversal es un
estadístico de sección cruzada) y por tanto no exhiben el prefijo de NaN que
aquella versión buscaba.

Las dos transformaciones que necesitan más de una serie se envuelven en una
función de una sola, recortando el panel auxiliar al mismo tramo que la serie
bajo contraste: comparar una pasada completa con la de un prefijo solo tiene
sentido si el resto de entradas se recorta igual.

In [ ]:
sectores = [a["nombre"] for a in catalogo["universo"] if a["rol"] == "sector"]


def vol_realizada_z(sp500):
    return features.zscore_causal(
        features.volatilidad_realizada(features.log_returns(sp500), ventana=21)
    )


def corr_accion_bono(sp500):
    tesoro = precios.loc[: sp500.index[-1], "tesoro_10y"]
    return features.correlacion_movil(
        features.log_returns(sp500), features.log_returns(tesoro), ventana=60
    )


def dispersion_sectorial(sp500):
    panel = precios.loc[: sp500.index[-1], sectores]
    return features.dispersion_transversal(features.log_returns(panel))


comprobaciones = [
    (features.log_returns, precios["sp500"], "ret_sp500"),
    (features.zscore_causal, precios["vix"], "vix_nivel_z"),
    (vol_realizada_z, precios["sp500"], "vol_realizada_z"),
    (features.drawdown, precios["sp500"], "drawdown_sp500"),
    (features.momentum, precios["sp500"], "momento_sp500"),
    (corr_accion_bono, precios["sp500"], "corr_accion_bono"),
    (dispersion_sectorial, precios["sp500"], "dispersion_sectorial"),
]

for transformacion, serie, nombre in comprobaciones:
    features.assert_causal(transformacion, serie, nombre)

print("Contraste de causalidad superado en", len(comprobaciones), "transformaciones.")

## Variables objetivo

Estas series sí miran al futuro, y es correcto: son la etiqueta, no una entrada.
El alineado con las ventanas de `X` lo garantiza `ventanas.construir_ventanas()`
en el notebook 02, que exige que la `X` de una muestra termine justo antes de
donde empieza su `Y`.

In [ ]:
v = config.ventanas()
objetivos = features.objetivos(precios, v.horizonte)

print("Horizonte:", v.horizonte, "días de mercado")
objetivos.describe().T

## Escritura

Parquet y no CSV: conserva los tipos y el índice de fechas sin ambigüedad de
formato, y pesa una fracción.

In [ ]:
src.DIR_PROCESADO.mkdir(parents=True, exist_ok=True)

canales.to_parquet(src.DIR_PROCESADO / "canales.parquet")
objetivos.to_parquet(src.DIR_PROCESADO / "objetivos.parquet")

print("Escritos", canales.shape[1], "canales y", objetivos.shape[1], "objetivos.")

## Salidas generadas

In [ ]:
from pathlib import Path

salidas = [
    src.DIR_CRUDO / "precios.parquet",
    src.DIR_PROCESADO / "canales.parquet",
    src.DIR_PROCESADO / "objetivos.parquet",
]

for ruta in salidas:
    ruta = Path(ruta)
    marca = "ok" if ruta.exists() else "--"
    print("[{}] {}".format(marca, ruta.relative_to(src.RAIZ)))
